## APARTADO 1: LIMPIEZA DE LOS DATOS Y PREPARACAÓN DE LOS DF.

### Importación de los paquetes

In [5]:
import pandas as pd
import numpy as np
import re
from sklearn.linear_model import LinearRegression

RUTA_DATOS = "./datasetsWC26/"     #carpeta con los ficheros csv originales
RUTA_SALIDA = "./datasets_limpios/"             #carpeta donde se exportan los .xlsx finales limpios

#### Función auxiliar para detectar el separador de cada csv

In [6]:
def separator(file_path: str) -> str:
    """Devuelve el separador (";" o ",") que delimita el archivo csv, leyendo su cabecera.
    Cada fichero del paquete usa un separador distinto, por eso se detecta automáticamente.
    Por defecto devuelve ",".
    """
    with open(file_path, mode="r", encoding="utf-8-sig") as f:
        header = f.readline()
    return ";" if ";" in header else ","


In [7]:
#definimos una función que realice un analisis básido inicial de los df

def analisis_inicial(df: pd.DataFrame):
    print(f'\n -> La forma del df es : {df.shape}')
    print(f'\n -> Las columnas son {df.columns.to_list()}')
    print(f'\n -> los formatos de las columnas son:')
    print(df.dtypes)
    print(f'\n -> Número de valores nulos: ')
    print(df.isna().sum())
    print(f'\n -> Número de lineas duplicadas {df.duplicated().sum()}')

    print(f' \n-> Análisis general con info: ')
    print(df.info())

    print(f' \n-> Métricas generales con describe {df.describe(include=["number"])}')

    print(f'\n-> Vista previa del df cabecera:')
    print(df.head())

    print(f'\n-> Vista previa del df final:')
    print(df.tail())



### PARTIDOS

empezamos importando matches_mundial2026.csv que servirá como base para el df partidos

In [8]:
partidos_raw = pd.read_csv(fr"{RUTA_DATOS}matches_mundial2026.csv", sep = separator(fr"{RUTA_DATOS}matches_mundial2026.csv"), na_values=["N/A"])

partidos_raw

,match_id,date,kickoff_time_utc,stage_id,venue_id,home_team_id,away_team_id,home_score,away_score,home_penalty_score,away_penalty_score,status,result_type,home_xg,away_xg,referee_id,player_of_the_match_id,capacidad_estadio_pct,clima
0,1,11/06/2026,19:00:00,1,1,1,2,2,0,NaN,NaN,Completed,Regular,184,52,42.0,16,63.1,soleado
1,2,11/06/2026,23:00:00,1,7,3,4,2,1,NaN,NaN,Completed,Regular,145,112,5.0,58,91.2,soleado
2,3,06/12/2026,19:00:00,1,6,5,6,1,1,NaN,NaN,Completed,Regular,135,98,15.0,112,NaN,NaN
3,4,12 June 2026,23:00:00,1,3,13,14,4,1,NaN,NaN,Completed,Regular,276,88,11.0,332,88.9,soleado
4,5,"June 13, 2026",15:00:00,1,10,7,8,1,1,NaN,NaN,Completed,Regular,78,154,38.0,157,NaN,soleado
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,60,26 de junio de 2026,22:00:00,1,16,22,23,1,1,NaN,NaN,Completed,Regular,112,95,8.0,557,130.5,Soleado
106,11,20260614,21:00:00,1,11,19,20,1,0,NaN,NaN,Completed,Regular,11,95,17.0,479,87.2,Nublado
107,35,20260620,21:00:00,1,14,21,23,5,1,NaN,NaN,Completed,Regular,345,88,29.0,531,74.6,DESPEJADO
108,20,2026-06-16T19:00:00Z,23:59:00,1,15,39,40,3,1,NaN,NaN,Completed,Regular,198,88,10.0,1023,61.0,Lluvia


observamos que hay 110 lineas, mientras que solo hubo 104 partidos en el mundial, por lo que habrá que ver si hay duplicados o lineas erroneas

In [9]:
print(f'ANÁLISIS INICIAL DEL DF PARTIDOS:')
analisis_inicial(partidos_raw)

ANÁLISIS INICIAL DEL DF PARTIDOS:

 -> La forma del df es : (110, 19)

 -> Las columnas son ['match_id', 'date', 'kickoff_time_utc', 'stage_id', 'venue_id', 'home_team_id', 'away_team_id', 'home_score', 'away_score', 'home_penalty_score', 'away_penalty_score', 'status', 'result_type', 'home_xg', 'away_xg', 'referee_id', 'player_of_the_match_id', 'capacidad_estadio_pct', 'clima']

 -> los formatos de las columnas son:
match_id                    int64
date                       object
kickoff_time_utc           object
stage_id                    int64
venue_id                    int64
home_team_id                int64
away_team_id                int64
home_score                  int64
away_score                  int64
home_penalty_score        float64
away_penalty_score        float64
status                     object
result_type                object
home_xg                     int64
away_xg                     int64
referee_id                float64
player_of_the_match_id      int64
c

eliminamos directamente las columnas que no vamos a necesitar.
Pero conservamos result_type que nos permitirá crear más adelante la columna prorroga, la eliminaremos después

In [10]:

partidos = partidos_raw.copy()
partidos = partidos.drop(columns=["kickoff_time_utc", "status", "home_xg", "away_xg", "player_of_the_match_id", "capacidad_estadio_pct", "clima" ])

mapeo_a_español_1_partidos={"match_id": "index_partido", "stage_id": "ronda", "venue_id": "estadio", "home_team_id": "seleccion_local", "away_team_id": "seleccion_visitante", "home_score": "marcador_local", "away_score": "marcador_visitante", "home_penalty_score": "penalty_local", "away_penalty_score": "penalty_visitante", "referee_id": "arbitro_partido"}
partidos = partidos.rename(columns = mapeo_a_español_1_partidos)
partidos.set_index("index_partido")
partidos.columns

Index(['index_partido', 'date', 'ronda', 'estadio', 'seleccion_local',
       'seleccion_visitante', 'marcador_local', 'marcador_visitante',
       'penalty_local', 'penalty_visitante', 'result_type', 'arbitro_partido'],
      dtype='object')

Gestión de duplicados

In [11]:

partidos[partidos.duplicated()]

,index_partido,date,ronda,estadio,seleccion_local,seleccion_visitante,marcador_local,marcador_visitante,penalty_local,penalty_visitante,result_type,arbitro_partido
104,33,20.06.2026,1,10,17,19,2,1,NaN,NaN,Regular,31.0
105,60,26 de junio de 2026,1,16,22,23,1,1,NaN,NaN,Regular,8.0
106,11,20260614,1,11,19,20,1,0,NaN,NaN,Regular,17.0
107,35,20260620,1,14,21,23,5,1,NaN,NaN,Regular,29.0
108,20,2026-06-16T19:00:00Z,1,15,39,40,3,1,NaN,NaN,Regular,10.0
109,10,Jun 14 2026,1,15,21,22,2,2,NaN,NaN,Regular,21.0


Gestión de duplicados: hay 6 lineas duplicadas que eliminar para que ya solo queden las 104 lineas de los 104 partidos

In [12]:
partidos.drop_duplicates(inplace=True)

In [13]:
#comprobar que todos los partidos del 1 al 104 estén en el df
indices_partidos = partidos["index_partido"].to_list()
algun_error = False

for i in range(1, 105):
    if i not in indices_partidos:
        print(f'El partido {i} no está en el df')
        algun_error = True

if not algun_error:
    print(f'Todos, los 104 los partidos, aparecen en el df partidos. ')

Todos, los 104 los partidos, aparecen en el df partidos. 


Gestión de los formatos erroneos en las fechas (expresiones regulares regex)

In [14]:
### bloque para la captura y conversion de las fechas
patron1 = re.compile(r"(\d{2})[/\.](\d{2})[/\.](2026)")               # DD/MM/YYYY  o  DD.MM.YYYY
patron_us = re.compile(r"(\d{2})-(\d{2})-(2026)")                     # MM-DD-YYYY
patron2 = re.compile(r"(\d{2})[\s-](\w{3,4})[\s-](\d{2,4})")          # 13-jun-26  /  12 June 2026
patron3 = re.compile(r"\s*(\w{3,4})\s+(\d{2}),?\s+(2026)\s*")         # June 13, 2026  /  Jun 14 2026
patron4 = re.compile(r"(\d{1,2})\sde\s(\w+)\sde\s(2026)")             # 14 de junio de 2026
patron5 = re.compile(r"(2026)(\d{2})(\d{2})")                         # 20260614
patron6 = re.compile(r"(2026)-(\d{2})-(\d{2})T19:00:00Z")             # 2026-06-13T19:00:00Z

def convertir_mes(mes_sucio):
    
    mapeo_de_meses = {"june": "06", "junio": "06", "jun": "06", "jul": "07", "july": "07", "julio": "07"}
    return mapeo_de_meses.get(mes_sucio.lower())

def aplicar_formato(row):
    fecha = row["date"]

    match1 = re.fullmatch(patron1, fecha)
    match_us = re.fullmatch(patron_us, fecha)
    match2 = re.fullmatch(patron2, fecha)
    match3 = re.fullmatch(patron3, fecha)
    match4 = re.fullmatch(patron4, fecha)
    match5 = re.fullmatch(patron5, fecha)
    match6 = re.fullmatch(patron6, fecha)


    if match1:
        dia, mes, año = match1.groups()
        fecha_limpia = f"2026-{mes}-{dia}"

    if match_us:
        mes, dia, año = match_us.groups()
        fecha_limpia = f"2026-{mes}-{dia}"

    elif match2:
        dia, mes, año = match2.groups()
        mes = convertir_mes(mes)
        fecha_limpia = f"2026-{mes}-{dia}"

    elif match3:
        mes, dia, año = match3.groups()
        mes = convertir_mes(mes)
        fecha_limpia = f"2026-{mes}-{dia}"

    elif match4:
        dia, mes, año = match4.groups()
        mes = convertir_mes(mes)
        fecha_limpia = f"2026-{mes}-{dia}"

    elif match5:
        año, mes, dia = match5.groups()
        fecha_limpia = f"2026-{mes}-{dia}"

    elif match6:
        año, mes, dia = match6.groups()
        fecha_limpia = f"2026-{mes}-{dia}"

    return fecha_limpia




In [15]:
partidos["fecha_partido"] = partidos.apply(aplicar_formato, axis = 1)
partidos["fecha_partido"] = pd.to_datetime(partidos["fecha_partido"])
partidos.drop(columns=["date"], inplace = True)
partidos

,index_partido,ronda,estadio,seleccion_local,seleccion_visitante,marcador_local,marcador_visitante,penalty_local,penalty_visitante,result_type,arbitro_partido,fecha_partido
0,1,1,1,1,2,2,0,NaN,NaN,Regular,42.0,2026-06-11
1,2,1,7,3,4,2,1,NaN,NaN,Regular,5.0,2026-06-11
2,3,1,6,5,6,1,1,NaN,NaN,Regular,15.0,2026-12-06
3,4,1,3,13,14,4,1,NaN,NaN,Regular,11.0,2026-06-12
4,5,1,10,7,8,1,1,NaN,NaN,Regular,38.0,2026-06-13
...,...,...,...,...,...,...,...,...,...,...,...,...
99,100,4,12,37,8,3,1,NaN,NaN,AET,26.0,2026-07-12
100,101,5,4,33,29,0,2,NaN,NaN,Regular,23.0,2026-07-14
101,102,5,9,45,37,1,2,NaN,NaN,Regular,21.0,2026-07-15
102,103,6,13,33,45,4,6,NaN,NaN,Regular,25.0,2026-07-18


Hay fechas que quedan fuera del periodo del mundial (11 jun - 19 jul 2026): son fechas escritas como MM/DD/YYYY que el patrón DD/MM ha leído con día y mes intercambiados. Como son solo 3, las podemos corregir a mano.

In [16]:
#eliminar los registros que no corresponden a las fechas del mundial
fecha_inicio = pd.to_datetime('2026-06-11')
fecha_fin = pd.to_datetime('2026-07-19')
partidos['fecha_partido'] = partidos['fecha_partido']

fuera_de_rango = partidos.loc[(partidos["fecha_partido"] < fecha_inicio) | (partidos["fecha_partido"] > fecha_fin)] 

fuera_de_rango

,index_partido,ronda,estadio,seleccion_local,seleccion_visitante,marcador_local,marcador_visitante,penalty_local,penalty_visitante,result_type,arbitro_partido,fecha_partido
2,3,1,6,5,6,1,1,NaN,NaN,Regular,15.0,2026-12-06
86,87,2,13,37,30,3,2,NaN,NaN,AET,8.0,2026-04-07
98,99,4,13,36,45,1,2,NaN,NaN,AET,8.0,2026-11-07


revisamos y corregimos esas 3 fechas

In [17]:
#arreglo a mano los 3 dias que se han leido mal y se han intercambiado día y mes

partidos.loc[2,"fecha_partido"] = pd.to_datetime("2026-06-12")
partidos.loc[86,"fecha_partido"] = pd.to_datetime("2026-07-04")
partidos.loc[98,"fecha_partido"] = pd.to_datetime("2026-07-11")

fuera_de_rango = partidos.loc[(partidos["fecha_partido"] < fecha_inicio) | (partidos["fecha_partido"] > fecha_fin)] 

fuera_de_rango



,index_partido,ronda,estadio,seleccion_local,seleccion_visitante,marcador_local,marcador_visitante,penalty_local,penalty_visitante,result_type,arbitro_partido,fecha_partido


In [18]:
partidos

,index_partido,ronda,estadio,seleccion_local,seleccion_visitante,marcador_local,marcador_visitante,penalty_local,penalty_visitante,result_type,arbitro_partido,fecha_partido
0,1,1,1,1,2,2,0,NaN,NaN,Regular,42.0,2026-06-11
1,2,1,7,3,4,2,1,NaN,NaN,Regular,5.0,2026-06-11
2,3,1,6,5,6,1,1,NaN,NaN,Regular,15.0,2026-06-12
3,4,1,3,13,14,4,1,NaN,NaN,Regular,11.0,2026-06-12
4,5,1,10,7,8,1,1,NaN,NaN,Regular,38.0,2026-06-13
...,...,...,...,...,...,...,...,...,...,...,...,...
99,100,4,12,37,8,3,1,NaN,NaN,AET,26.0,2026-07-12
100,101,5,4,33,29,0,2,NaN,NaN,Regular,23.0,2026-07-14
101,102,5,9,45,37,1,2,NaN,NaN,Regular,21.0,2026-07-15
102,103,6,13,33,45,4,6,NaN,NaN,Regular,25.0,2026-07-18


FORMATEAR

In [19]:


partidos["penalty_local"] = partidos["penalty_local"].astype("Int64")
partidos["arbitro_partido"] = partidos["arbitro_partido"].astype("Int64")
partidos["penalty_visitante"] = partidos["penalty_visitante"].astype("Int64")

print(partidos.dtypes)

index_partido                   int64
ronda                           int64
estadio                         int64
seleccion_local                 int64
seleccion_visitante             int64
marcador_local                  int64
marcador_visitante              int64
penalty_local                   Int64
penalty_visitante               Int64
result_type                    object
arbitro_partido                 Int64
fecha_partido          datetime64[ns]
dtype: object


### importamos ahora los arbitros de index_referees.csv para asociarlos a los partidos

In [20]:
arbitros_raw = pd.read_csv(fr"{RUTA_DATOS}index_referees.csv", sep = separator(fr"{RUTA_DATOS}index_referees.csv"))
arbitros_raw.head(5)

,referee_id,name,country,avg_cards_per_game,referee_id.1
0,1,Abdulrahman Al-Jassim,Jordan,42,1
1,2,Adham Makhadmeh,China,55,2
2,2,Adham Makhadmeh,China,55,2
3,3,Alejandro José Hernández,Argentina,50,3
4,4,Alireza Faghani,Portugal,47,4


In [21]:
analisis_inicial(arbitros_raw)


 -> La forma del df es : (49, 5)

 -> Las columnas son ['referee_id', 'name', 'country', 'avg_cards_per_game', 'referee_id.1']

 -> los formatos de las columnas son:
referee_id             int64
name                  object
country               object
avg_cards_per_game     int64
referee_id.1           int64
dtype: object

 -> Número de valores nulos: 
referee_id            0
name                  0
country               0
avg_cards_per_game    0
referee_id.1          0
dtype: int64

 -> Número de lineas duplicadas 6
 
-> Análisis general con info: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49 entries, 0 to 48
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   referee_id          49 non-null     int64 
 1   name                49 non-null     object
 2   country             49 non-null     object
 3   avg_cards_per_game  49 non-null     int64 
 4   referee_id.1        49 non-null     int64 
dtyp

In [22]:
print(f'En este df hay {arbitros_raw.duplicated().sum()} duplicados. Los eliminamos.')
print(arbitros_raw[arbitros_raw.duplicated()])
arbitros = arbitros_raw.drop_duplicates()

En este df hay 6 duplicados. Los eliminamos.
    referee_id                name    country  avg_cards_per_game  \
2            2     Adham Makhadmeh      China                  55   
8            7  César Arturo Ramos  Argentina                  50   
26          24         Jalal Jayed     Mexico                  40   
29          26       João Pinheiro  Argentina                  54   
37          33         Omar Al Ali     Norway                  29   
45          40    Szymon Marciniak    Morocco                  46   

    referee_id.1  
2              2  
8              7  
26            24  
29            26  
37            33  
45            40  


hacemos un merge para cruzar los nombres de los arbitros con los partidos

In [23]:
partidos = pd.merge(partidos.reset_index(), arbitros[["referee_id", "name"]], how='left', left_on="arbitro_partido", right_on="referee_id").set_index("index_partido")
partidos.drop(columns = ["index", "referee_id", "arbitro_partido"], inplace = True)
partidos = partidos.rename(columns = {"name": "arbitro_partido"})
print(f'Tras el merge, comprobamos que se ha realizado bien porque sigue habiendo {partidos.shape[0]} lineas.')

Tras el merge, comprobamos que se ha realizado bien porque sigue habiendo 104 lineas.


el árbitro de ese partido no figura en los datos originales (referee_id vacío) y no hay forma de recuperarlo, así que lo marcamos explícitamente como `Desconocido` en lugar de dejar un nulo.

In [24]:
partidos[partidos["arbitro_partido"].isna()]

,ronda,estadio,seleccion_local,seleccion_visitante,marcador_local,marcador_visitante,penalty_local,penalty_visitante,result_type,fecha_partido,arbitro_partido
index_partido,,,,,,,,,,,
53,1,2,12,9,0,3,<NA>,<NA>,Regular,2026-06-25,NaN


In [25]:
partidos["arbitro_partido"] = partidos["arbitro_partido"].fillna("Desconocido")

In [26]:
partidos

,ronda,estadio,seleccion_local,seleccion_visitante,marcador_local,marcador_visitante,penalty_local,penalty_visitante,result_type,fecha_partido,arbitro_partido
index_partido,,,,,,,,,,,
1,1,1,1,2,2,0,<NA>,<NA>,Regular,2026-06-11,Wilton Sampaio
2,1,7,3,4,2,1,<NA>,<NA>,Regular,2026-06-11,Amin Omar
3,1,6,5,6,1,1,<NA>,<NA>,Regular,2026-06-12,Facundo Tello
4,1,3,13,14,4,1,<NA>,<NA>,Regular,2026-06-12,Danny Makkelie
5,1,10,7,8,1,1,<NA>,<NA>,Regular,2026-06-13,Saíd Martínez
...,...,...,...,...,...,...,...,...,...,...,...
100,4,12,37,8,3,1,<NA>,<NA>,AET,2026-07-12,João Pinheiro
101,5,4,33,29,0,2,<NA>,<NA>,Regular,2026-07-14,Iván Barton
102,5,9,45,37,1,2,<NA>,<NA>,Regular,2026-07-15,Ismail Elfath


### importamos ahora las fases de tournament_stages.csv

In [27]:
tournament_stages_raw = pd.read_csv(fr"{RUTA_DATOS}tournament_stages.csv", sep = separator(fr"{RUTA_DATOS}tournament_stages.csv"))
tournament_stages_raw.head(9)

,stage_id,stage_name,is_knockout,num_partidos,fecha_inicio_fase
0,1,Group Stage,False,104.0,11/06/2026
1,2,Round of 32,True,32.0,2026-06-28
2,3,Round of 16,True,NaN,"July 4, 2026"
3,4,Quarter-finals,True,8.0,NaN
4,5,Semi-finals,True,4.0,09-jul-2026
5,6,Third-place match,True,999.0,14/07/2026
6,7,Final,True,1.0,19-07-2026
7,3,Round of 16,True,NaN,"July 4, 2026"
8,6,Third-place match,True,999.0,14/07/2026


In [28]:
analisis_inicial(tournament_stages_raw)


 -> La forma del df es : (9, 5)

 -> Las columnas son ['stage_id', 'stage_name', 'is_knockout', 'num_partidos', 'fecha_inicio_fase']

 -> los formatos de las columnas son:
stage_id               int64
stage_name            object
is_knockout             bool
num_partidos         float64
fecha_inicio_fase     object
dtype: object

 -> Número de valores nulos: 
stage_id             0
stage_name           0
is_knockout          0
num_partidos         2
fecha_inicio_fase    1
dtype: int64

 -> Número de lineas duplicadas 2
 
-> Análisis general con info: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   stage_id           9 non-null      int64  
 1   stage_name         9 non-null      object 
 2   is_knockout        9 non-null      bool   
 3   num_partidos       7 non-null      float64
 4   fecha_inicio_fase  8 non-null      object 
dtype

In [29]:
print(f'En este df hay {tournament_stages_raw.duplicated().sum()} duplicados. Los eliminamos.')
print(tournament_stages_raw[tournament_stages_raw.duplicated()])
tournament_stages = tournament_stages_raw.drop_duplicates()
tournament_stages["stage_name"] = tournament_stages["stage_name"].str.strip()
tournament_stages

En este df hay 2 duplicados. Los eliminamos.
   stage_id         stage_name  is_knockout  num_partidos fecha_inicio_fase
7         3        Round of 16         True           NaN      July 4, 2026
8         6  Third-place match         True         999.0        14/07/2026


/var/folders/lx/9synkbf173d0sq0bh4fdj7sc0000gn/T/ipykernel_22003/1420470358.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tournament_stages["stage_name"] = tournament_stages["stage_name"].str.strip()


,stage_id,stage_name,is_knockout,num_partidos,fecha_inicio_fase
0,1,Group Stage,False,104.0,11/06/2026
1,2,Round of 32,True,32.0,2026-06-28
2,3,Round of 16,True,NaN,"July 4, 2026"
3,4,Quarter-finals,True,8.0,NaN
4,5,Semi-finals,True,4.0,09-jul-2026
5,6,Third-place match,True,999.0,14/07/2026
6,7,Final,True,1.0,19-07-2026


In [30]:
#sustituimos el id de la fase por su nombre
partidos = pd.merge(partidos.reset_index(), tournament_stages[["stage_id", "stage_name"]], how='left', left_on="ronda", right_on="stage_id").set_index("index_partido")
partidos.drop(columns = ["stage_id", "ronda"], inplace = True)
partidos = partidos.rename(columns = {"stage_name": "ronda"})
print(f'Tras el merge, comprobamos que se ha realizado bien porque sigue habiendo {partidos.shape[0]} lineas.')
partidos

Tras el merge, comprobamos que se ha realizado bien porque sigue habiendo 104 lineas.


,estadio,seleccion_local,seleccion_visitante,marcador_local,marcador_visitante,penalty_local,penalty_visitante,result_type,fecha_partido,arbitro_partido,ronda
index_partido,,,,,,,,,,,
1,1,1,2,2,0,<NA>,<NA>,Regular,2026-06-11,Wilton Sampaio,Group Stage
2,7,3,4,2,1,<NA>,<NA>,Regular,2026-06-11,Amin Omar,Group Stage
3,6,5,6,1,1,<NA>,<NA>,Regular,2026-06-12,Facundo Tello,Group Stage
4,3,13,14,4,1,<NA>,<NA>,Regular,2026-06-12,Danny Makkelie,Group Stage
5,10,7,8,1,1,<NA>,<NA>,Regular,2026-06-13,Saíd Martínez,Group Stage
...,...,...,...,...,...,...,...,...,...,...,...
100,12,37,8,3,1,<NA>,<NA>,AET,2026-07-12,João Pinheiro,Quarter-finals
101,4,33,29,0,2,<NA>,<NA>,Regular,2026-07-14,Iván Barton,Semi-finals
102,9,45,37,1,2,<NA>,<NA>,Regular,2026-07-15,Ismail Elfath,Semi-finals


### importamos ahora los estadios de worldcup_venues.csv

In [31]:
worldcup_venues_raw = pd.read_csv(fr"{RUTA_DATOS}worldcup_venues.csv", sep = separator(fr"{RUTA_DATOS}worldcup_venues.csv"))
worldcup_venues_raw

,venue_id,stadium_name,city,country,capacity,elevation_meters,fecha_inauguracion,superficie_cesped
0,1,Mexico City Stadium (Estadio Azteca),Mexico City,MEX,80824,2200.0,1966,Híbrido
1,2,New York New Jersey Stadium (MetLife Stadium),East Rutherford,USA,82500,7.0,2010-08-01,NaN
2,3,Los Angeles Stadium (SoFi Stadium),Inglewood,usa,70240,30.0,1987,Artificial
3,4,Dallas Stadium (AT&T Stadium),Arlington,USA,80000,184.0,NaN,Natural
4,5,Vancouver Stadium (BC Place),Vancouver,CAN,54500,9.0,2020,Natural
5,6,Toronto Stadium (BMO Field),Toronto,CAN,-1,76.0,14/05/1993,HIBRIDO
6,7,Guadalajara Stadium (Estadio Akron),Zapopan,MEX,48070,1566.0,1998,Natural
7,8,Monterrey Stadium (Estadio BBVA),Guadalupe,MEX,53500,NaN,circa 1970,natural
8,9,Atlanta Stadium (Mercedes-Benz Stadium),Atlanta,USA,71000,318.0,2003,Híbrido
9,10,Boston Stadium (Gillette Stadium),Foxborough,USA,500000,87.0,NaN,natural


In [32]:
analisis_inicial(worldcup_venues_raw)


 -> La forma del df es : (19, 8)

 -> Las columnas son ['venue_id', 'stadium_name', 'city', 'country', 'capacity', 'elevation_meters', 'fecha_inauguracion', 'superficie_cesped']

 -> los formatos de las columnas son:
venue_id                int64
stadium_name           object
city                   object
country                object
capacity                int64
elevation_meters      float64
fecha_inauguracion     object
superficie_cesped      object
dtype: object

 -> Número de valores nulos: 
venue_id              0
stadium_name          0
city                  0
country               0
capacity              0
elevation_meters      1
fecha_inauguracion    4
superficie_cesped     4
dtype: int64

 -> Número de lineas duplicadas 3
 
-> Análisis general con info: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   venue_id           

In [33]:
print(f'En este df hay {worldcup_venues_raw.duplicated().sum()} duplicados. Los eliminamos.')
print(worldcup_venues_raw[worldcup_venues_raw.duplicated()])
worldcup_venues = worldcup_venues_raw.drop_duplicates()
worldcup_venues

En este df hay 3 duplicados. Los eliminamos.
    venue_id                          stadium_name         city country  \
16         1  Mexico City Stadium (Estadio Azteca)  Mexico City     MEX   
17        10     Boston Stadium (Gillette Stadium)   Foxborough     USA   
18         6           Toronto Stadium (BMO Field)      Toronto     CAN   

    capacity  elevation_meters fecha_inauguracion superficie_cesped  
16     80824            2200.0               1966           Híbrido  
17    500000              87.0                NaN           natural  
18        -1              76.0         14/05/1993           HIBRIDO  


,venue_id,stadium_name,city,country,capacity,elevation_meters,fecha_inauguracion,superficie_cesped
0,1,Mexico City Stadium (Estadio Azteca),Mexico City,MEX,80824,2200.0,1966,Híbrido
1,2,New York New Jersey Stadium (MetLife Stadium),East Rutherford,USA,82500,7.0,2010-08-01,NaN
2,3,Los Angeles Stadium (SoFi Stadium),Inglewood,usa,70240,30.0,1987,Artificial
3,4,Dallas Stadium (AT&T Stadium),Arlington,USA,80000,184.0,NaN,Natural
4,5,Vancouver Stadium (BC Place),Vancouver,CAN,54500,9.0,2020,Natural
5,6,Toronto Stadium (BMO Field),Toronto,CAN,-1,76.0,14/05/1993,HIBRIDO
6,7,Guadalajara Stadium (Estadio Akron),Zapopan,MEX,48070,1566.0,1998,Natural
7,8,Monterrey Stadium (Estadio BBVA),Guadalupe,MEX,53500,NaN,circa 1970,natural
8,9,Atlanta Stadium (Mercedes-Benz Stadium),Atlanta,USA,71000,318.0,2003,Híbrido
9,10,Boston Stadium (Gillette Stadium),Foxborough,USA,500000,87.0,NaN,natural


In [34]:
#observamos que USA está escrito con mayusculas y con minusculas
print(worldcup_venues["country"].unique())
print(f'corregimos escritura')
worldcup_venues["country"] = worldcup_venues["country"].replace("usa", "USA")
print(worldcup_venues["country"].unique())

['MEX' 'USA' 'usa' 'CAN']
corregimos escritura
['MEX' 'USA' 'CAN']


/var/folders/lx/9synkbf173d0sq0bh4fdj7sc0000gn/T/ipykernel_22003/1647402837.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  worldcup_venues["country"] = worldcup_venues["country"].replace("usa", "USA")


In [35]:
partidos = pd.merge(partidos.reset_index(), worldcup_venues[["venue_id", "stadium_name", "city", "country"]], how='left', left_on="estadio", right_on="venue_id").set_index("index_partido")
partidos.drop(columns = ["venue_id", "estadio"], inplace = True)
partidos = partidos.rename(columns = {"city": "ciudad_partido", "country": "pais_partido", "stadium_name": "estadio"})
print(f'Tras el merge, comprobamos que se ha realizado bien porque sigue habiendo {partidos.shape[0]} lineas.')
partidos

Tras el merge, comprobamos que se ha realizado bien porque sigue habiendo 104 lineas.


,seleccion_local,seleccion_visitante,marcador_local,marcador_visitante,penalty_local,penalty_visitante,result_type,fecha_partido,arbitro_partido,ronda,estadio,ciudad_partido,pais_partido
index_partido,,,,,,,,,,,,,
1,1,2,2,0,<NA>,<NA>,Regular,2026-06-11,Wilton Sampaio,Group Stage,Mexico City Stadium (Estadio Azteca),Mexico City,MEX
2,3,4,2,1,<NA>,<NA>,Regular,2026-06-11,Amin Omar,Group Stage,Guadalajara Stadium (Estadio Akron),Zapopan,MEX
3,5,6,1,1,<NA>,<NA>,Regular,2026-06-12,Facundo Tello,Group Stage,Toronto Stadium (BMO Field),Toronto,CAN
4,13,14,4,1,<NA>,<NA>,Regular,2026-06-12,Danny Makkelie,Group Stage,Los Angeles Stadium (SoFi Stadium),Inglewood,USA
5,7,8,1,1,<NA>,<NA>,Regular,2026-06-13,Saíd Martínez,Group Stage,Boston Stadium (Gillette Stadium),Foxborough,USA
...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,37,8,3,1,<NA>,<NA>,AET,2026-07-12,João Pinheiro,Quarter-finals,Kansas City Stadium (Arrowhead Stadium),Kansas City,USA
101,33,29,0,2,<NA>,<NA>,Regular,2026-07-14,Iván Barton,Semi-finals,Dallas Stadium (AT&T Stadium),Arlington,USA
102,45,37,1,2,<NA>,<NA>,Regular,2026-07-15,Ismail Elfath,Semi-finals,Atlanta Stadium (Mercedes-Benz Stadium),Atlanta,USA


Columna `prorroga`: usamos `result_type` (`AET` = ganó en la prórroga, `Penalties` = llegó a penaltis, que implica prórroga previa). Hay un partido que sorprende, el partdo 84 entre protugal y croacia, que sin tener prorroga tiene un evento en el 103, esto se debe a un tiempo añadido de 20 minutos (hubo muchas polemicas en el partido), pero es correcto.

In [36]:
partidos["prorroga"] = ((partidos["result_type"] == "AET") |(partidos["result_type"] == "Penalties"))
partidos["prorroga"] = partidos["prorroga"].map({True: "SI", False: "NO"})
partidos.drop(columns=["result_type"], inplace = True)
print(f'\nPartidos con prórroga: {(partidos["prorroga"] == "SI").sum()}')


Partidos con prórroga: 9


### JUGADORES

importamos los datos de los jugadores 2026player_stats.csv que servirá de base para el df jugadores

In [37]:
jugadores_raw = pd.read_csv(fr"{RUTA_DATOS}2026player_stats.csv", sep = separator(fr"{RUTA_DATOS}2026player_stats.csv"))
jugadores_raw

,player_id,player_name,team_id,position,matches_played,matches_started,minutes_played,goals,assists,shots,shots_on_target,yellow_cards,red_cards,penalty_goals,own_goals,clean_sheets,saves,goals_conceded,Unnamed: 18,average_rating
0,1,JosŽ Raœl Rangel,1,GK,5,5,438,0.0,0.0,0.0,NaN,0.0,0.0,0,0,4.0,7.0,3.0,NaN,7.20
1,2,Jorge Eduardo Sanchez,1,DEF,4,4,349,0.0,1.0,1.0,NaN,1.0,0.0,0,0,NaN,NaN,NaN,NaN,6.68
2,3,CŽsar Jasib Montes,1,DEF,4,4,315,0.0,0.0,1.0,NaN,0.0,1.0,0,0,NaN,NaN,NaN,NaN,7.05
3,4,Edson Omar Alvarez,1,DEF,4,2,239,0.0,0.0,2.0,NaN,1.0,0.0,0,0,NaN,NaN,NaN,NaN,7.08
4,5,Johan Felipe Vasquez,1,DEF,4,4,360,0.0,0.0,1.0,NaN,1.0,0.0,0,0,NaN,NaN,NaN,NaN,7.08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1243,1244,Orlando Mosquera,48,GK,3,3,270,0.0,0.0,0.0,NaN,0.0,0.0,0,0,0.0,8.0,4.0,NaN,6.80
1244,1245,Michael Amir Murillo,48,DEF,3,3,270,0.0,0.0,5.0,NaN,0.0,0.0,0,0,NaN,NaN,NaN,NaN,6.93
1245,1246,Azar’as Emmanuel Londono,48,FWD,3,0,53,0.0,0.0,1.0,NaN,0.0,0.0,0,0,NaN,NaN,NaN,NaN,6.57
1246,1247,Roderick Alonso Miller,48,DEF,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN


In [38]:
analisis_inicial(jugadores_raw)
#no hay duplicados



 -> La forma del df es : (1248, 20)

 -> Las columnas son ['player_id', 'player_name', 'team_id', 'position', 'matches_played', 'matches_started', 'minutes_played', 'goals', 'assists', 'shots', 'shots_on_target', 'yellow_cards', 'red_cards', 'penalty_goals', 'own_goals', 'clean_sheets', 'saves', 'goals_conceded', 'Unnamed: 18', 'average_rating']

 -> los formatos de las columnas son:
player_id            int64
player_name         object
team_id              int64
position            object
matches_played       int64
matches_started      int64
minutes_played       int64
goals              float64
assists            float64
shots              float64
shots_on_target    float64
yellow_cards       float64
red_cards          float64
penalty_goals        int64
own_goals            int64
clean_sheets       float64
saves              float64
goals_conceded     float64
Unnamed: 18        float64
average_rating     float64
dtype: object

 -> Número de valores nulos: 
player_id             0
pla

In [39]:
jugadores_raw.columns

Index(['player_id', 'player_name', 'team_id', 'position', 'matches_played',
       'matches_started', 'minutes_played', 'goals', 'assists', 'shots',
       'shots_on_target', 'yellow_cards', 'red_cards', 'penalty_goals',
       'own_goals', 'clean_sheets', 'saves', 'goals_conceded', 'Unnamed: 18',
       'average_rating'],
      dtype='object')

eliminamos directamente las columnas que no vamos a necesitar
quitamos tambien `player_name`porque vienen los nombrs con una codificacion mal escrita (`JosŽ Raœl Rangel`), preferiremos quedarnos con la del csv 2026squads_and_players.csv (`José Raúl Rangel`)

In [40]:


jugadores = jugadores_raw.copy()

jugadores = jugadores.drop(columns= ["player_name", "shots_on_target","matches_started","penalty_goals","own_goals","clean_sheets", "Unnamed: 18"])
mapeo_a_español_2_jugadores={"player_name": "nombre_jugador", "position": "posicion", "matches_played": "partidos_jugados", "minutes_played": "minutos_jugados", "goals": "goles", "assists": "asistencias", "shots": "tiros", "yellow_cards": "tarjetas_amarillas", "red_cards": "tarjetas_rojas", "saves": "paradas", "goals_conceded":"goles_concedidos", "average_rating":"calificacion_media"}
jugadores = jugadores.rename(columns = mapeo_a_español_2_jugadores)

jugadores

,player_id,team_id,posicion,partidos_jugados,minutos_jugados,goles,asistencias,tiros,tarjetas_amarillas,tarjetas_rojas,paradas,goles_concedidos,calificacion_media
0,1,1,GK,5,438,0.0,0.0,0.0,0.0,0.0,7.0,3.0,7.20
1,2,1,DEF,4,349,0.0,1.0,1.0,1.0,0.0,NaN,NaN,6.68
2,3,1,DEF,4,315,0.0,0.0,1.0,0.0,1.0,NaN,NaN,7.05
3,4,1,DEF,4,239,0.0,0.0,2.0,1.0,0.0,NaN,NaN,7.08
4,5,1,DEF,4,360,0.0,0.0,1.0,1.0,0.0,NaN,NaN,7.08
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1243,1244,48,GK,3,270,0.0,0.0,0.0,0.0,0.0,8.0,4.0,6.80
1244,1245,48,DEF,3,270,0.0,0.0,5.0,0.0,0.0,NaN,NaN,6.93
1245,1246,48,FWD,3,53,0.0,0.0,1.0,0.0,0.0,NaN,NaN,6.57
1246,1247,48,DEF,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [42]:
#ninguno de los player_id es nulo lo cual es util para futuros merge
jugadores.isna().sum()

player_id                0
team_id                  0
posicion                 0
partidos_jugados         0
minutos_jugados          0
goles                  213
asistencias            213
tiros                  320
tarjetas_amarillas     213
tarjetas_rojas         213
paradas               1103
goles_concedidos      1103
calificacion_media     320
dtype: int64

### importamos los datos de los jugadores 2026squads_and_players.csv

In [43]:
squads_players_raw = pd.read_csv(fr"{RUTA_DATOS}2026squads_and_players.csv", sep = separator(fr"{RUTA_DATOS}2026squads_and_players.csv"))
squads_players_raw

,player_id,team_id,player_name,position,club_team,market_value_eur,caps,international_goals
0,1,1,José Raúl Rangel,GK,CD Guadalajara,8.9336,15,0
1,2,1,Jorge Eduardo Sanchez,DEF,PAOK Saloniki,24.6286,59,3
2,3,1,César Jasib Montes,DEF,FC Lokomotiv Moscow,10.9126,69,4
3,4,1,Edson Omar Alvarez,DEF,Fenerbahçe SK,10.8759,100,7
4,5,1,Johan Felipe Vasquez,DEF,Genoa CFC,18.2654,47,3
...,...,...,...,...,...,...,...,...
1243,1244,48,Orlando Mosquera,GK,Al Fayha FC,0.7000,50,0
1244,1245,48,Michael Amir Murillo,DEF,Be ş ikta ş JK,7.8302,94,9
1245,1246,48,Azarías Emmanuel Londono,FWD,CD Universidad Católica,4.4904,13,1
1246,1247,48,Roderick Alonso Miller,DEF,Turan Tovuz,NaN,49,2


In [44]:
analisis_inicial(squads_players_raw)
#no hay duplicados ni nulos en los valores identificativos y coincide el tamaño con el número de jugadores que hobo en el mundial


 -> La forma del df es : (1248, 8)

 -> Las columnas son ['player_id', 'team_id', 'player_name', 'position', 'club_team', 'market_value_eur', 'caps', 'international_goals']

 -> los formatos de las columnas son:
player_id                int64
team_id                  int64
player_name             object
position                object
club_team               object
market_value_eur       float64
caps                     int64
international_goals      int64
dtype: object

 -> Número de valores nulos: 
player_id               0
team_id                 0
player_name             0
position                0
club_team               0
market_value_eur       90
caps                    0
international_goals     0
dtype: int64

 -> Número de lineas duplicadas 0
 
-> Análisis general con info: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1248 entries, 0 to 1247
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  

In [45]:
jugadores

,player_id,team_id,posicion,partidos_jugados,minutos_jugados,goles,asistencias,tiros,tarjetas_amarillas,tarjetas_rojas,paradas,goles_concedidos,calificacion_media
0,1,1,GK,5,438,0.0,0.0,0.0,0.0,0.0,7.0,3.0,7.20
1,2,1,DEF,4,349,0.0,1.0,1.0,1.0,0.0,NaN,NaN,6.68
2,3,1,DEF,4,315,0.0,0.0,1.0,0.0,1.0,NaN,NaN,7.05
3,4,1,DEF,4,239,0.0,0.0,2.0,1.0,0.0,NaN,NaN,7.08
4,5,1,DEF,4,360,0.0,0.0,1.0,1.0,0.0,NaN,NaN,7.08
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1243,1244,48,GK,3,270,0.0,0.0,0.0,0.0,0.0,8.0,4.0,6.80
1244,1245,48,DEF,3,270,0.0,0.0,5.0,0.0,0.0,NaN,NaN,6.93
1245,1246,48,FWD,3,53,0.0,0.0,1.0,0.0,0.0,NaN,NaN,6.57
1246,1247,48,DEF,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [46]:

jugadores = pd.merge(jugadores, squads_players_raw[["player_id", "player_name", "club_team", "market_value_eur", "caps"]], how='left', on="player_id")


jugadores = jugadores.rename(columns = {"club_team":"club_jugador", "market_value_eur": "valor_mercado", "player_name": "nombre_jugador", "caps": "partidos_seleccion"})
print(f'Tras el merge, comprobamos que se ha realizado bien porque sigue habiendo {jugadores.shape[0]} lineas.')

jugadores


Tras el merge, comprobamos que se ha realizado bien porque sigue habiendo 1248 lineas.


,player_id,team_id,posicion,partidos_jugados,minutos_jugados,goles,asistencias,tiros,tarjetas_amarillas,tarjetas_rojas,paradas,goles_concedidos,calificacion_media,nombre_jugador,club_jugador,valor_mercado,partidos_seleccion
0,1,1,GK,5,438,0.0,0.0,0.0,0.0,0.0,7.0,3.0,7.20,José Raúl Rangel,CD Guadalajara,8.9336,15
1,2,1,DEF,4,349,0.0,1.0,1.0,1.0,0.0,NaN,NaN,6.68,Jorge Eduardo Sanchez,PAOK Saloniki,24.6286,59
2,3,1,DEF,4,315,0.0,0.0,1.0,0.0,1.0,NaN,NaN,7.05,César Jasib Montes,FC Lokomotiv Moscow,10.9126,69
3,4,1,DEF,4,239,0.0,0.0,2.0,1.0,0.0,NaN,NaN,7.08,Edson Omar Alvarez,Fenerbahçe SK,10.8759,100
4,5,1,DEF,4,360,0.0,0.0,1.0,1.0,0.0,NaN,NaN,7.08,Johan Felipe Vasquez,Genoa CFC,18.2654,47
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1243,1244,48,GK,3,270,0.0,0.0,0.0,0.0,0.0,8.0,4.0,6.80,Orlando Mosquera,Al Fayha FC,0.7000,50
1244,1245,48,DEF,3,270,0.0,0.0,5.0,0.0,0.0,NaN,NaN,6.93,Michael Amir Murillo,Be ş ikta ş JK,7.8302,94
1245,1246,48,FWD,3,53,0.0,0.0,1.0,0.0,0.0,NaN,NaN,6.57,Azarías Emmanuel Londono,CD Universidad Católica,4.4904,13
1246,1247,48,DEF,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Roderick Alonso Miller,Turan Tovuz,NaN,49


### importamos ahora las selecciones de 2026nationalteams.csv

In [47]:
selecciones_raw = pd.read_csv(fr"{RUTA_DATOS}2026nationalteams.csv", sep = separator(fr"{RUTA_DATOS}2026nationalteams.csv"))
selecciones_raw
#de entrada se ve que hay 63 rows pero solo 48 equipos queparticiparon

,team_id,team_name,fifa_code,group_letter,confederation,fifa_ranking_pre_tournament,elo_rating,manager_name
0,1.0,Mexico,MEX,A,CONCACAF,14,1810,Javier Aguirre
1,2.0,South Africa,RSA,A,UNKNOWN,60,1620,Hugo Broos
2,3.0,South Korea,KOR,0,NaN,22,1800,Hong Myung-bo
3,4.0,Czechia,CZE,A,UEFA,40,NaN,Miroslav Koubek
4,5.0,Canada,CAN,B,CONCACAF,33,2500,Jesse Marsch
...,...,...,...,...,...,...,...,...
58,NaN,Zanzibar,NaN,J,NaN,NaN,1300,NaN
59,NaN,Tibet,NaN,NaN,AFC,999,NaN,T. Unknown
60,NaN,Real Madrid,NaN,A,UEFA,NaN,NaN,NaN
61,NaN,Manchester City,NaN,A,NaN,CLUB,CLUB,Club Manager


In [48]:
analisis_inicial(selecciones_raw)
#no hay duplicados


 -> La forma del df es : (63, 8)

 -> Las columnas son ['team_id', 'team_name', 'fifa_code', 'group_letter', 'confederation', 'fifa_ranking_pre_tournament', 'elo_rating', 'manager_name']

 -> los formatos de las columnas son:
team_id                        float64
team_name                       object
fifa_code                       object
group_letter                    object
confederation                   object
fifa_ranking_pre_tournament     object
elo_rating                      object
manager_name                    object
dtype: object

 -> Número de valores nulos: 
team_id                         7
team_name                       0
fifa_code                       7
group_letter                    5
confederation                   7
fifa_ranking_pre_tournament     9
elo_rating                     10
manager_name                    8
dtype: int64

 -> Número de lineas duplicadas 8
 
-> Análisis general con info: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63 entries, 0

In [49]:
selecciones_raw[selecciones_raw.duplicated()]

,team_id,team_name,fifa_code,group_letter,confederation,fifa_ranking_pre_tournament,elo_rating,manager_name
48,9.0,Brazil,BRA,C,CONMEBOL,6,2030,Carlo Ancelotti
49,5.0,Canada,CAN,B,CONCACAF,33,2500,Jesse Marsch
50,23.0,Sweden,SWE,F,UEFA,25,1815,Jon Dahl Tomasson
51,28.0,New Zealand,NZL,G,OFC,85,NaN,Darren Bazeley
52,6.0,Bosnia and Herzegovina,BIH,B,UEFA,64,1645,Sergej Barbarez
53,24.0,Tunisia,TUN,F,CAF,NaN,1710,Hervé Renard
54,4.0,Czechia,CZE,A,UEFA,40,NaN,Miroslav Koubek
55,3.0,South Korea,KOR,0,NaN,22,1800,Hong Myung-bo


In [50]:
#eliminamos filas duplicadas
selecciones = selecciones_raw[["team_id", "team_name"]].drop_duplicates()
selecciones.shape

(55, 2)

In [51]:
#hay 7 selecciones cuyo team id es nulo NaN
selecciones[selecciones["team_id"].isna()]

,team_id,team_name
56,NaN,Greenland
57,NaN,Catalonia
58,NaN,Zanzibar
59,NaN,Tibet
60,NaN,Real Madrid
61,NaN,Manchester City
62,NaN,Bayern Munich


In [52]:
#eliminamos con una mascara aquellos equipos que no tienen id
mask = ~selecciones["team_id"].isna()
selecciones = selecciones.loc[mask, ["team_id", "team_name"]]
print(selecciones.shape)

(48, 2)


In [53]:
selecciones.head(10)

,team_id,team_name
0,1.0,Mexico
1,2.0,South Africa
2,3.0,South Korea
3,4.0,Czechia
4,5.0,Canada
5,6.0,Bosnia and Herzegovina
6,7.0,Qatar
7,8.0,Switzerland
8,9.0,Brazil
9,10.0,Morocco


In [54]:
#quedan ya solo 48 lineas validad sin NaN en sus id y sin duplicados, coincide con el numero real de selecciones en el mundial, y esta listo apra el merge
selecciones.shape

(48, 2)

In [55]:
jugadores = pd.merge(jugadores, selecciones, how='left', on="team_id")

jugadores.drop(columns = ["team_id",], inplace = True)
jugadores.rename(columns = {"team_name": "seleccion_nacional"}, inplace=True)

print(f'Tras el merge, comprobamos que se ha realizado bien porque sigue habiendo {jugadores.shape[0]} lineas.')
jugadores.set_index("nombre_jugador", inplace = True )
jugadores

Tras el merge, comprobamos que se ha realizado bien porque sigue habiendo 1248 lineas.


,player_id,posicion,partidos_jugados,minutos_jugados,goles,asistencias,tiros,tarjetas_amarillas,tarjetas_rojas,paradas,goles_concedidos,calificacion_media,club_jugador,valor_mercado,partidos_seleccion,seleccion_nacional
nombre_jugador,,,,,,,,,,,,,,,,
José Raúl Rangel,1,GK,5,438,0.0,0.0,0.0,0.0,0.0,7.0,3.0,7.20,CD Guadalajara,8.9336,15,Mexico
Jorge Eduardo Sanchez,2,DEF,4,349,0.0,1.0,1.0,1.0,0.0,NaN,NaN,6.68,PAOK Saloniki,24.6286,59,Mexico
César Jasib Montes,3,DEF,4,315,0.0,0.0,1.0,0.0,1.0,NaN,NaN,7.05,FC Lokomotiv Moscow,10.9126,69,Mexico
Edson Omar Alvarez,4,DEF,4,239,0.0,0.0,2.0,1.0,0.0,NaN,NaN,7.08,Fenerbahçe SK,10.8759,100,Mexico
Johan Felipe Vasquez,5,DEF,4,360,0.0,0.0,1.0,1.0,0.0,NaN,NaN,7.08,Genoa CFC,18.2654,47,Mexico
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Orlando Mosquera,1244,GK,3,270,0.0,0.0,0.0,0.0,0.0,8.0,4.0,6.80,Al Fayha FC,0.7000,50,Panama
Michael Amir Murillo,1245,DEF,3,270,0.0,0.0,5.0,0.0,0.0,NaN,NaN,6.93,Be ş ikta ş JK,7.8302,94,Panama
Azarías Emmanuel Londono,1246,FWD,3,53,0.0,0.0,1.0,0.0,0.0,NaN,NaN,6.57,CD Universidad Católica,4.4904,13,Panama


In [56]:
partidos

,seleccion_local,seleccion_visitante,marcador_local,marcador_visitante,penalty_local,penalty_visitante,fecha_partido,arbitro_partido,ronda,estadio,ciudad_partido,pais_partido,prorroga
index_partido,,,,,,,,,,,,,
1,1,2,2,0,<NA>,<NA>,2026-06-11,Wilton Sampaio,Group Stage,Mexico City Stadium (Estadio Azteca),Mexico City,MEX,NO
2,3,4,2,1,<NA>,<NA>,2026-06-11,Amin Omar,Group Stage,Guadalajara Stadium (Estadio Akron),Zapopan,MEX,NO
3,5,6,1,1,<NA>,<NA>,2026-06-12,Facundo Tello,Group Stage,Toronto Stadium (BMO Field),Toronto,CAN,NO
4,13,14,4,1,<NA>,<NA>,2026-06-12,Danny Makkelie,Group Stage,Los Angeles Stadium (SoFi Stadium),Inglewood,USA,NO
5,7,8,1,1,<NA>,<NA>,2026-06-13,Saíd Martínez,Group Stage,Boston Stadium (Gillette Stadium),Foxborough,USA,NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,37,8,3,1,<NA>,<NA>,2026-07-12,João Pinheiro,Quarter-finals,Kansas City Stadium (Arrowhead Stadium),Kansas City,USA,SI
101,33,29,0,2,<NA>,<NA>,2026-07-14,Iván Barton,Semi-finals,Dallas Stadium (AT&T Stadium),Arlington,USA,NO
102,45,37,1,2,<NA>,<NA>,2026-07-15,Ismail Elfath,Semi-finals,Atlanta Stadium (Mercedes-Benz Stadium),Atlanta,USA,NO


In [57]:
#añadir la columna de seleccion local a partidos
partidos = pd.merge(partidos.reset_index(), selecciones, how = "left", left_on="seleccion_local", right_on="team_id").set_index("index_partido")
partidos.drop(columns = ["seleccion_local", "team_id"], inplace=True)
partidos.rename(columns = {"team_name":"seleccion_local"}, inplace=True)

partidos

,seleccion_visitante,marcador_local,marcador_visitante,penalty_local,penalty_visitante,fecha_partido,arbitro_partido,ronda,estadio,ciudad_partido,pais_partido,prorroga,seleccion_local
index_partido,,,,,,,,,,,,,
1,2,2,0,<NA>,<NA>,2026-06-11,Wilton Sampaio,Group Stage,Mexico City Stadium (Estadio Azteca),Mexico City,MEX,NO,Mexico
2,4,2,1,<NA>,<NA>,2026-06-11,Amin Omar,Group Stage,Guadalajara Stadium (Estadio Akron),Zapopan,MEX,NO,South Korea
3,6,1,1,<NA>,<NA>,2026-06-12,Facundo Tello,Group Stage,Toronto Stadium (BMO Field),Toronto,CAN,NO,Canada
4,14,4,1,<NA>,<NA>,2026-06-12,Danny Makkelie,Group Stage,Los Angeles Stadium (SoFi Stadium),Inglewood,USA,NO,USA
5,8,1,1,<NA>,<NA>,2026-06-13,Saíd Martínez,Group Stage,Boston Stadium (Gillette Stadium),Foxborough,USA,NO,Qatar
...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,8,3,1,<NA>,<NA>,2026-07-12,João Pinheiro,Quarter-finals,Kansas City Stadium (Arrowhead Stadium),Kansas City,USA,SI,Argentina
101,29,0,2,<NA>,<NA>,2026-07-14,Iván Barton,Semi-finals,Dallas Stadium (AT&T Stadium),Arlington,USA,NO,France
102,37,1,2,<NA>,<NA>,2026-07-15,Ismail Elfath,Semi-finals,Atlanta Stadium (Mercedes-Benz Stadium),Atlanta,USA,NO,England


In [58]:
#añadir la columna de seleccion visitante a partidos
partidos = pd.merge(partidos.reset_index(), selecciones, how = "left", left_on="seleccion_visitante", right_on="team_id").set_index("index_partido")
partidos.drop(columns = ["seleccion_visitante", "team_id"], inplace=True)
partidos.rename(columns = {"team_name":"seleccion_visitante"}, inplace=True)

partidos

,marcador_local,marcador_visitante,penalty_local,penalty_visitante,fecha_partido,arbitro_partido,ronda,estadio,ciudad_partido,pais_partido,prorroga,seleccion_local,seleccion_visitante
index_partido,,,,,,,,,,,,,
1,2,0,<NA>,<NA>,2026-06-11,Wilton Sampaio,Group Stage,Mexico City Stadium (Estadio Azteca),Mexico City,MEX,NO,Mexico,South Africa
2,2,1,<NA>,<NA>,2026-06-11,Amin Omar,Group Stage,Guadalajara Stadium (Estadio Akron),Zapopan,MEX,NO,South Korea,Czechia
3,1,1,<NA>,<NA>,2026-06-12,Facundo Tello,Group Stage,Toronto Stadium (BMO Field),Toronto,CAN,NO,Canada,Bosnia and Herzegovina
4,4,1,<NA>,<NA>,2026-06-12,Danny Makkelie,Group Stage,Los Angeles Stadium (SoFi Stadium),Inglewood,USA,NO,USA,Paraguay
5,1,1,<NA>,<NA>,2026-06-13,Saíd Martínez,Group Stage,Boston Stadium (Gillette Stadium),Foxborough,USA,NO,Qatar,Switzerland
...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,3,1,<NA>,<NA>,2026-07-12,João Pinheiro,Quarter-finals,Kansas City Stadium (Arrowhead Stadium),Kansas City,USA,SI,Argentina,Switzerland
101,0,2,<NA>,<NA>,2026-07-14,Iván Barton,Semi-finals,Dallas Stadium (AT&T Stadium),Arlington,USA,NO,France,Spain
102,1,2,<NA>,<NA>,2026-07-15,Ismail Elfath,Semi-finals,Atlanta Stadium (Mercedes-Benz Stadium),Atlanta,USA,NO,England,Argentina


In [59]:
#ya están todos los datos que necesitamos en el df jugadores, ahora tenemos que formatearlo
jugadores.dtypes

player_id               int64
posicion               object
partidos_jugados        int64
minutos_jugados         int64
goles                 float64
asistencias           float64
tiros                 float64
tarjetas_amarillas    float64
tarjetas_rojas        float64
paradas               float64
goles_concedidos      float64
calificacion_media    float64
club_jugador           object
valor_mercado         float64
partidos_seleccion      int64
seleccion_nacional     object
dtype: object

In [60]:
for columna in ["goles", "asistencias", "tiros", "tarjetas_amarillas", "tarjetas_rojas", "paradas", "goles_concedidos"]:
    jugadores[columna] = jugadores[columna].astype('Int64')

jugadores.dtypes

player_id               int64
posicion               object
partidos_jugados        int64
minutos_jugados         int64
goles                   Int64
asistencias             Int64
tiros                   Int64
tarjetas_amarillas      Int64
tarjetas_rojas          Int64
paradas                 Int64
goles_concedidos        Int64
calificacion_media    float64
club_jugador           object
valor_mercado         float64
partidos_seleccion      int64
seleccion_nacional     object
dtype: object

#### Revisión de valores anómalos y de nulos en jugadores

In [61]:
#chequeo de la validez de los valores (que sean positivos)
for columna in ["partidos_jugados", "minutos_jugados", "goles", "asistencias", "tiros", "tarjetas_amarillas", "tarjetas_rojas", "paradas", "goles_concedidos"]:
    print(f'\n-> {columna}')
    print(f"\tValores_nulos: {jugadores[columna].isna().sum()}")

    if jugadores[jugadores[columna]<0].shape[0] > 0:
        print(f'\tValores negativos : {jugadores[jugadores[columna]<0]}')
    else:
        print(f'\tValores negativos : 0')



-> partidos_jugados
	Valores_nulos: 0
	Valores negativos : 0

-> minutos_jugados
	Valores_nulos: 0
	Valores negativos : 0

-> goles
	Valores_nulos: 213
	Valores negativos : 0

-> asistencias
	Valores_nulos: 213
	Valores negativos : 0

-> tiros
	Valores_nulos: 320
	Valores negativos : 0

-> tarjetas_amarillas
	Valores_nulos: 213
	Valores negativos : 0

-> tarjetas_rojas
	Valores_nulos: 213
	Valores negativos : 0

-> paradas
	Valores_nulos: 1103
	Valores negativos : 0

-> goles_concedidos
	Valores_nulos: 1103
	Valores negativos : 0


In [62]:
#checkeo de validez de las calificacion
print(jugadores["calificacion_media"].isna().sum())
mascara=(jugadores["calificacion_media"]<0) | (jugadores["calificacion_media"]>10)
jugadores[mascara]

320


,player_id,posicion,partidos_jugados,minutos_jugados,goles,asistencias,tiros,tarjetas_amarillas,tarjetas_rojas,paradas,goles_concedidos,calificacion_media,club_jugador,valor_mercado,partidos_seleccion,seleccion_nacional
nombre_jugador,,,,,,,,,,,,,,,,


no hay ningún jugador con notas fuera del rango 0-10 pero sí bastantes que no tienen calificacion

In [63]:
#chequeo de la validez de las posiciones: todas son o portero o defensa. medio o delamntero
jugadores["posicion"].unique()

array(['GK', 'DEF', 'MID', 'FWD'], dtype=object)

In [64]:
selecciones_posibles = selecciones["team_name"].to_list()
error_seleccion = False

for seleccion in jugadores["seleccion_nacional"]:
    if seleccion not in selecciones_posibles:
        print(f'La seleccion {seleccion} no está entre las permitidas')
        error_seleccion = True

if not error_seleccion:
    print(f'No hay selecciones prohibidas')

No hay selecciones prohibidas


Penaltis de las tandas: los valores vienen multiplicados por 10 (20, 30, 40 en lugar de 2, 3, 4), lo comprobamos y los dividimos. Además, el enunciado pide `0 si no aplica`, así que los partidos sin tanda de penaltis (NaN) pasan a 0.

In [65]:
partidos["penalty_local"].unique()

<IntegerArray>
[<NA>, 30, 20, 40]
Length: 4, dtype: Int64

In [66]:
partidos["penalty_visitante"].unique()

<IntegerArray>
[<NA>, 40, 30]
Length: 3, dtype: Int64

In [67]:
partidos["penalty_visitante"] = (partidos["penalty_visitante"] / 10).astype("Int64")
partidos["penalty_local"] = (partidos["penalty_local"] / 10).astype("Int64")
partidos["penalty_visitante"] = partidos["penalty_visitante"].fillna(0)
partidos["penalty_local"] = partidos["penalty_local"].fillna(0)

In [68]:
partidos["penalty_visitante"].unique()

<IntegerArray>
[0, 4, 3]
Length: 3, dtype: Int64

In [69]:
partidos["penalty_local"].unique()

<IntegerArray>
[0, 3, 2, 4]
Length: 4, dtype: Int64

hemos comprobado: el formato de todas las columnas, que los valores de las estadisticas sean factibles, no sean negativos, para las calificaciones, que estén entre 0-10, y que las posiciones de los jugadores estén entre las permitidas, y que las selecciones de los jugadores estén también entre las permitidas
hemos limpiado duplicados, hecho los merge pertinentes, convertido a los formatos que queriamos etc

#### curiosidad, los 20 futbolistas mejor valorados del mundial y los 20 que mas goles han metido

In [70]:
jugadores["calificacion_media"].sort_values(ascending=False).head(20)

nombre_jugador
Lionel Andrés Messi                8.79
Lawrence Ati Zigi                  8.45
Kylian Mbappe                      8.24
Mateo Chavez                       8.10
José Vinicius                      8.00
Giovani Lo Celso                   7.90
Thelonious Gerard Aasgaard         7.90
Erling Braut Haaland               7.90
Hazem Mastouri                     7.90
Orlando Daniel Gill                7.88
José Vozinha                       7.88
Elijah Henry Just                  7.83
Jude Victor William Bellingham     7.81
Amad Diallo                        7.78
Rodrigo Rodri                      7.76
Julián Andrés Quinones             7.72
Stephen Eustaquio                  7.70
Lewis Ferguson                     7.67
Jacob Mikael Widell Zetterstrom    7.65
Paul Jan-Paul Van Hecke            7.65
Name: calificacion_media, dtype: float64

In [71]:
jugadores["goles"].sort_values(ascending=False).head(20)

nombre_jugador
Kylian Mbappe                           10
Lionel Andrés Messi                      8
Jude Victor William Bellingham           7
Erling Braut Haaland                     7
Harry Edward Kane                        6
Masour Ousmane Dembele                   6
Mikel Oyarzabal                          5
Julián Andrés Quinones                   4
José Vinicius                            4
Ismaila Sarr                             4
Cody Mathès Gakpo                        3
Bradley Jean-Manuel Essolisa Barcola     3
Deniz Undav                              3
Yoane Wissa                              3
Lautaro Javier Martinez                  3
Jonathan Christian David                 3
Bukayo Ayoyinka Saka                     3
Folarin Jolaoluwa Balogun                3
Ismael Saibari                           3
Ronaldo Cristiano Ronaldo                3
Name: goles, dtype: Int64

### importamos ahora 2026match_register.csv para procesar los eventos en los partidos

In [72]:
eventos_raw = pd.read_csv(fr"{RUTA_DATOS}2026match_register.csv", sep = separator(fr"{RUTA_DATOS}2026match_register.csv"))
eventos_raw

,event_id,match_id,minute,event_type,team_id,player_id
0,1,1.0,9.0,Goal,1.0,16.0
1,2,1.0,9.0,Assist,1.0,6.0
2,3,1.0,17.0,Yellow Card,2.0,30.0
3,4,1.0,23.0,Yellow Card,1.0,26.0
4,5,1.0,55.0,Red Card,2.0,39.0
...,...,...,...,...,...,...
859,41,4.0,301.0,Assist,13.0,328.0
860,206,27.0,350.0,Goal,5.0,182.0
861,138,18.0,420.0,Goal,36.0,919.0
862,698,91.0,500.0,Goal,36.0,919.0


In [73]:
print(f'ANÁLISIS INICIAL DEL DF PARTIDOS:')
analisis_inicial(eventos_raw)

ANÁLISIS INICIAL DEL DF PARTIDOS:

 -> La forma del df es : (864, 6)

 -> Las columnas son ['event_id', 'match_id', 'minute', 'event_type', 'team_id', 'player_id']

 -> los formatos de las columnas son:
event_id        int64
match_id      float64
minute        float64
event_type     object
team_id       float64
player_id     float64
dtype: object

 -> Número de valores nulos: 
event_id       0
match_id      15
minute        53
event_type    15
team_id       15
player_id     15
dtype: int64

 -> Número de lineas duplicadas 20
 
-> Análisis general con info: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 864 entries, 0 to 863
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   event_id    864 non-null    int64  
 1   match_id    849 non-null    float64
 2   minute      811 non-null    float64
 3   event_type  849 non-null    object 
 4   team_id     849 non-null    float64
 5   player_id   849 non-null    float64
dtyp

In [74]:
eventos = eventos_raw.copy()
eventos = eventos.drop(columns = ["team_id"])
eventos.drop_duplicates(inplace=True)
eventos.rename(columns={"event_id": "index_evento", "match_id": "index_partido", "minute": "minuto_evento", "event_type": "tipo_evento"}, inplace=True)
eventos

,index_evento,index_partido,minuto_evento,tipo_evento,player_id
0,1,1.0,9.0,Goal,16.0
1,2,1.0,9.0,Assist,6.0
2,3,1.0,17.0,Yellow Card,30.0
3,4,1.0,23.0,Yellow Card,26.0
4,5,1.0,55.0,Red Card,39.0
...,...,...,...,...,...
859,41,4.0,301.0,Assist,328.0
860,206,27.0,350.0,Goal,182.0
861,138,18.0,420.0,Goal,919.0
862,698,91.0,500.0,Goal,919.0


In [75]:
eventos["index_partido"] = eventos["index_partido"].astype("Int64")
eventos["minuto_evento"] = eventos["minuto_evento"].astype("Int64")
eventos["player_id"] = eventos["player_id"].astype("Int64")

In [76]:
#eliminar los eventos asociados a partidos que no existen
mascara = (eventos["index_partido"] >= 1) & (eventos["index_partido"] <= 104)
eventos = eventos.loc[mascara]
eventos

,index_evento,index_partido,minuto_evento,tipo_evento,player_id
0,1,1,9,Goal,16
1,2,1,9,Assist,6
2,3,1,17,Yellow Card,30
3,4,1,23,Yellow Card,26
4,5,1,55,Red Card,39
...,...,...,...,...,...
859,41,4,301,Assist,328
860,206,27,350,Goal,182
861,138,18,420,Goal,919
862,698,91,500,Goal,919


In [77]:
eventos[(eventos["minuto_evento"] < 0)]

,index_evento,index_partido,minuto_evento,tipo_evento,player_id
108,109,14,-5,Yellow Card,645
122,123,16,-2,Goal,684
160,161,21,-10,Yellow Card,1042
266,267,35,-45,Assist,542
531,532,70,-1,Yellow Card,1094


In [78]:
eventos[(eventos["minuto_evento"] > 120)]

,index_evento,index_partido,minuto_evento,tipo_evento,player_id
577,578,75,121,Penalty Shootout Miss,423
579,580,75,121,Penalty Shootout Goal,422
580,581,75,121,Penalty Shootout Goal,353
581,582,75,121,Penalty Shootout Goal,426
582,583,75,121,Penalty Shootout Goal,361
583,584,75,121,Penalty Shootout Miss,427
585,586,75,121,Penalty Shootout Goal,436
586,587,75,121,Penalty Shootout Miss,343
588,589,75,121,Penalty Shootout Goal,351
595,596,76,121,Penalty Shootout Goal,540


observamos que en el minuto 121 se apuntan las tandas de penaltis, por lo que tenemos quue ocnservar ese minuto también!!!

In [79]:
#eliminar los eventos asociados a minutos más allá del 121 (no existen tampoco) OJO QUE LOS DEL 121 son los penaltis
print(eventos.shape)
mascara = (
    (eventos["minuto_evento"] >= 0) & (eventos["minuto_evento"] <= 121)
) | eventos["minuto_evento"].isna()
eventos = eventos.loc[mascara]
eventos.shape

(834, 5)


(824, 5)

se han eliminado los 10 eventos que correspondian a minutos negativos y más alla del 121 (300, 500, 999 centinela etc)

In [80]:
print(eventos.dtypes)
print(f'Convertimos los tipos a int')
eventos["index_partido"] = eventos["index_partido"].astype('Int64')
eventos["minuto_evento"] = eventos["minuto_evento"].astype('Int64')
eventos["player_id"] = eventos["player_id"].astype('Int64')
print(eventos.dtypes)

index_evento      int64
index_partido     Int64
minuto_evento     Int64
tipo_evento      object
player_id         Int64
dtype: object
Convertimos los tipos a int
index_evento      int64
index_partido     Int64
minuto_evento     Int64
tipo_evento      object
player_id         Int64
dtype: object


/var/folders/lx/9synkbf173d0sq0bh4fdj7sc0000gn/T/ipykernel_22003/2757093378.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eventos["index_partido"] = eventos["index_partido"].astype('Int64')
/var/folders/lx/9synkbf173d0sq0bh4fdj7sc0000gn/T/ipykernel_22003/2757093378.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eventos["minuto_evento"] = eventos["minuto_evento"].astype('Int64')
/var/folders/lx/9synkbf173d0sq0bh4fdj7sc0000gn/T/ipykernel_22003/2757093378.py:5: SettingWithCopyWarning: 
A value is tr

In [81]:
#asegurarse de que los jugadores presentes en la tabla existen, es decir, que su index esté entre 1 y 1248  -> mo hay ninguno: es correcto.
eventos[eventos["player_id"]<1]

,index_evento,index_partido,minuto_evento,tipo_evento,player_id


In [82]:
eventos[eventos["player_id"]>1248]

,index_evento,index_partido,minuto_evento,tipo_evento,player_id


In [83]:
#nos damos cuenta que las revisiones del VAR vienen dadas con los str diferentes, y las unificamos en 1
print(eventos["tipo_evento"].value_counts())
print(f'\n Vamos a unificar VAR Review y Review VAR en una única.')
eventos["tipo_evento"] = eventos["tipo_evento"].str.strip().replace("Review VAR", "VAR Review")
eventos["tipo_evento"].value_counts()


tipo_evento
Goal                     306
Yellow Card              250
Assist                   200
Penalty Shootout Goal     24
Red Card                  15
Penalty Shootout Miss     14
VAR Review                 9
Review VAR                 6
Name: count, dtype: int64

 Vamos a unificar VAR Review y Review VAR en una única.


/var/folders/lx/9synkbf173d0sq0bh4fdj7sc0000gn/T/ipykernel_22003/3646300972.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eventos["tipo_evento"] = eventos["tipo_evento"].str.strip().replace("Review VAR", "VAR Review")


tipo_evento
Goal                     306
Yellow Card              250
Assist                   200
Penalty Shootout Goal     24
Red Card                  15
VAR Review                15
Penalty Shootout Miss     14
Name: count, dtype: int64

In [84]:
print(f'Ya tenemos limpio y validado el df de match_register')
eventos

Ya tenemos limpio y validado el df de match_register


,index_evento,index_partido,minuto_evento,tipo_evento,player_id
0,1,1,9,Goal,16
1,2,1,9,Assist,6
2,3,1,17,Yellow Card,30
3,4,1,23,Yellow Card,26
4,5,1,55,Red Card,39
...,...,...,...,...,...
829,830,104,93,Red Card,960
830,831,104,106,Goal,735
831,832,104,106,Assist,745
832,833,104,111,Yellow Card,956


In [85]:
#hacemos un merge para rellenar con los datos de nombre del jugador y seleccion
eventos = pd.merge(eventos, jugadores.reset_index()[["player_id", "nombre_jugador", "seleccion_nacional"]])
eventos.drop(columns=["player_id"], inplace=True)
eventos.rename(columns={"seleccion_nacional":"seleccion_evento"}, inplace=True)
eventos

,index_evento,index_partido,minuto_evento,tipo_evento,nombre_jugador,seleccion_evento
0,1,1,9,Goal,Julián Andrés Quinones,Mexico
1,2,1,9,Assist,Erik Antonio Lira,Mexico
2,3,1,17,Yellow Card,Teboho Mokoena,South Africa
3,4,1,23,Yellow Card,Brian Gutierrez,Mexico
4,5,1,55,Red Card,Sphephelo S'miso Sithole,South Africa
...,...,...,...,...,...,...
819,830,104,93,Red Card,Enzo Jeremías Fernandez,Argentina
820,831,104,106,Goal,Ferran Torres,Spain
821,832,104,106,Assist,Nicholas Williams,Spain
822,833,104,111,Yellow Card,Alexis Mac Allister,Argentina


In [86]:
jugadores

,player_id,posicion,partidos_jugados,minutos_jugados,goles,asistencias,tiros,tarjetas_amarillas,tarjetas_rojas,paradas,goles_concedidos,calificacion_media,club_jugador,valor_mercado,partidos_seleccion,seleccion_nacional
nombre_jugador,,,,,,,,,,,,,,,,
José Raúl Rangel,1,GK,5,438,0,0,0,0,0,7,3,7.20,CD Guadalajara,8.9336,15,Mexico
Jorge Eduardo Sanchez,2,DEF,4,349,0,1,1,1,0,<NA>,<NA>,6.68,PAOK Saloniki,24.6286,59,Mexico
César Jasib Montes,3,DEF,4,315,0,0,1,0,1,<NA>,<NA>,7.05,FC Lokomotiv Moscow,10.9126,69,Mexico
Edson Omar Alvarez,4,DEF,4,239,0,0,2,1,0,<NA>,<NA>,7.08,Fenerbahçe SK,10.8759,100,Mexico
Johan Felipe Vasquez,5,DEF,4,360,0,0,1,1,0,<NA>,<NA>,7.08,Genoa CFC,18.2654,47,Mexico
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Orlando Mosquera,1244,GK,3,270,0,0,0,0,0,8,4,6.80,Al Fayha FC,0.7000,50,Panama
Michael Amir Murillo,1245,DEF,3,270,0,0,5,0,0,<NA>,<NA>,6.93,Be ş ikta ş JK,7.8302,94,Panama
Azarías Emmanuel Londono,1246,FWD,3,53,0,0,1,0,0,<NA>,<NA>,6.57,CD Universidad Católica,4.4904,13,Panama


verificacion de los NA de columnas de estadisticas individuales, en muchas se PODRÍA poner los nan a 0, pero no lo vamos a hacer de momento porque no queremos perder la información (por ejemplo un jugador de campo va a tener siempre nan paradas porque solo puede hacer paradas un portero, realmente no es que su valor de la estadistica de paradas sea 0, porque nopuede hacerlas, sino nan; lo mismo pasa para los jugadores que no estñan jugando, no tienen un valor 0, porque estan en el banquillo, sino nan)

In [87]:
print(jugadores["tarjetas_amarillas"].value_counts(dropna=False))
print(f'Tras comprobar las bases oficiales del mundial, y como es lógico, encaja que aquellos registros de tarjetas amarillas que aparecen como NA sean 0')

tarjetas_amarillas
0       812
<NA>    213
1       194
2        28
3         1
Name: count, dtype: Int64
Tras comprobar las bases oficiales del mundial, y como es lógico, encaja que aquellos registros de tarjetas amarillas que aparecen como NA sean 0


In [88]:
print(jugadores["tarjetas_rojas"].value_counts(dropna=False))
print(f'Tras comprobar las bases oficiales del mundial, y como es lógico, encaja que aquellos registros de tarjetas rojas que aparecen como NA sean 0.')

tarjetas_rojas
0       1020
<NA>     213
1         15
Name: count, dtype: Int64
Tras comprobar las bases oficiales del mundial, y como es lógico, encaja que aquellos registros de tarjetas rojas que aparecen como NA sean 0.


In [89]:
print(jugadores["asistencias"].value_counts(dropna=False))
print(jugadores["paradas"].value_counts(dropna=False))
print(jugadores["goles_concedidos"].value_counts(dropna=False))
print(f'Todos ellos encajan con los datos reales de la web de la FIFA y podríamos poner a 0 los nan cuando sea necesario para la imputacion por regresion')

asistencias
0       887
<NA>    213
1       114
2        22
3         7
4         4
8         1
Name: count, dtype: Int64
paradas
<NA>    1103
0         87
8          7
7          7
12         6
9          5
11         4
16         4
18         3
23         3
4          3
17         2
25         2
6          2
5          2
20         1
14         1
10         1
29         1
13         1
15         1
3          1
19         1
Name: count, dtype: Int64
goles_concedidos
<NA>    1103
0         86
4         11
3          9
6          9
5          9
8          7
7          5
10         3
1          3
9          2
2          1
Name: count, dtype: Int64
Todos ellos encajan con los datos reales de la web de la FIFA y podríamos poner a 0 los nan cuando sea necesario para la imputacion por regresion


### Imputación de los datos que están como NAN

In [90]:
jugadores.isna().sum()

player_id                0
posicion                 0
partidos_jugados         0
minutos_jugados          0
goles                  213
asistencias            213
tiros                  320
tarjetas_amarillas     213
tarjetas_rojas         213
paradas               1103
goles_concedidos      1103
calificacion_media     320
club_jugador             0
valor_mercado           90
partidos_seleccion       0
seleccion_nacional       0
dtype: int64

In [91]:
eventos.isna().sum()

index_evento         0
index_partido        0
minuto_evento       38
tipo_evento          0
nombre_jugador       0
seleccion_evento     0
dtype: int64

#### vamos a imputar el valor_mercado por la media de la seleccion del jugador, calificacion_media por regresión, minuto_evento por interpolación

imputamos los valores nulos del valos_mercado de los jugadores por la media de los valores de mercado de sus compañeros de selección

In [ ]:
# un valor de mercado 0.0 no es real, es un dato ausente: lo pasamos a NaN para que también se impute
jugadores["valor_mercado"] = jugadores["valor_mercado"].replace(0.0, np.nan)

jugadores["valor_mercado"] = jugadores["valor_mercado"].fillna(jugadores.groupby("seleccion_nacional")["valor_mercado"].transform("mean"))
jugadores["valor_mercado"].isna().sum()

np.int64(0)

Calificaciones iguales a 0.0: son 18 jugadores que solo han jugado 1 o 2 minutos. Una nota de 0 sobre 10 no es una valoración real (la media del torneo es ~6.6) sino un valor centinela por no tener minutos suficientes, y distorsionaría tanto la regresión como la calificación media de cada selección. Por eso se consideran anómalos: los pasamos a NaN para que se imputen por regresión como el resto de ausentes.

In [ ]:
print(f'Jugadores con calificacion_media = 0.0 (anómalos): {(jugadores["calificacion_media"] == 0).sum()}')
jugadores.loc[jugadores["calificacion_media"] == 0, "calificacion_media"] = np.nan
print(f'Nulos en calificacion_media antes de imputar: {jugadores["calificacion_media"].isna().sum()}')

Calificacion_media: imputamos los datos faltantes con Imputación por Regresión basada en la correlación con las estadísticas de goles, asistencias, tarjetas
amarillas y rojas, paradas o goles concedidos (Distiguimos por posición del jugador: no es lo mismo un delantero que un portero)

PORTEROS

In [93]:
### hago la regresion de la calificacion media para los porteros GK

#generamos una copia de los datos y SOLO EN ESA COPIA sustotuyo los nan de las estadisticas personales por 0, que ya hemos visto que se podia en este caso pero luego en el df principal de jugadores seguiremos manteniedo es dato nan
datos_modelo = jugadores.copy()
columnas_de_entrenamiento = ["goles", "asistencias", "tarjetas_amarillas", "tarjetas_rojas", "paradas", "goles_concedidos"]
datos_modelo[columnas_de_entrenamiento] = (datos_modelo[columnas_de_entrenamiento].fillna(0)) #si no se rellenan los valores de las columnas de enyrenamiento por nulos (que en este caso sí se puede como vimos antes, explotara el modelo de regresión)

#crear los dartos de entrenamiento y entrenar
mascara_portero = (jugadores["posicion"]=="GK")

train = datos_modelo.loc[mascara_portero].copy()
train = train.dropna(subset=["calificacion_media"])

model = LinearRegression().fit(train[columnas_de_entrenamiento], train[["calificacion_media"]])

#imputar las calificaciones ausentes
mask_imputar = (mascara_portero & jugadores["calificacion_media"].isna())

predicciones = model.predict(datos_modelo.loc[mask_imputar, columnas_de_entrenamiento])
predicciones = np.clip(predicciones, 0, 10).round(2)
jugadores.loc[mask_imputar, "calificacion_media"] = predicciones



RESTO DE POSICIONES (no les aplica ya ni paradas ni goles concedidios que es solo para los porteros)

In [94]:
jugadores["posicion"].unique()

array(['GK', 'DEF', 'MID', 'FWD'], dtype=object)

In [ ]:
### hago la regresion de la calificacion media para los porteros GK

for posicion in ["DEF", "MID", "FWD"]:
    #generamos una copia de los datos y SOLO EN ESA COPIA sustotuyo los nan de las estadisticas personales por 0, que ya hemos visto que se podia en este caso pero luego en el df principal de jugadores seguiremos manteniedo es dato nan
    datos_modelo = jugadores.copy()
    columnas_de_entrenamiento = ["goles", "asistencias", "tarjetas_amarillas", "tarjetas_rojas"]
    datos_modelo[columnas_de_entrenamiento] = (datos_modelo[columnas_de_entrenamiento].fillna(0)) #si no se rellenan los valores de las columnas de enyrenamiento por nulos (que en este caso sí se puede como vimos antes, explotara el modelo de regresión)

    #crear los dartos de entrenamiento y entrenar
    mascara_posicion = (jugadores["posicion"]==posicion)

    train = datos_modelo.loc[mascara_posicion].copy()
    train = train.dropna(subset=["calificacion_media"])

    model = LinearRegression().fit(train[columnas_de_entrenamiento], train[["calificacion_media"]])

    #imputar las calificaciones ausentes
    mask_imputar = (mascara_posicion & jugadores["calificacion_media"].isna())

    predicciones = model.predict(datos_modelo.loc[mask_imputar, columnas_de_entrenamiento])
    predicciones = np.clip(predicciones, 0, 10).round(2)
    jugadores.loc[mask_imputar, "calificacion_media"] = predicciones



In [96]:
mascara_portero = (jugadores["posicion"]=="GK")
jugadores[mascara_portero]

,player_id,posicion,partidos_jugados,minutos_jugados,goles,asistencias,tiros,tarjetas_amarillas,tarjetas_rojas,paradas,goles_concedidos,calificacion_media,club_jugador,valor_mercado,partidos_seleccion,seleccion_nacional
nombre_jugador,,,,,,,,,,,,,,,,
José Raúl Rangel,1,GK,5,438,0,0,0,0,0,7,3,7.20,CD Guadalajara,8.933600,15,Mexico
Carlos Acevedo,12,GK,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,0,0,5.93,Club Santos Laguna,15.432000,7,Mexico
Francisco Guillermo Ochoa,13,GK,1,12,0,0,0,0,0,0,0,6.50,AEL Limassol,11.314500,153,Mexico
Ronwen Hayden Williams,27,GK,4,360,0,0,0,0,0,17,4,7.15,Mamelodi Sundowns FC,0.800000,65,South Africa
Sipho Justin Chaine,42,GK,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,0,0,5.93,Orlando Pirates FC,1.600000,4,South Africa
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Joseph Tetteh Anang,1208,GK,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,0,0,5.93,St Patrick's Athletic FC,0.350000,1,Ghana
Benjamin Asare,1212,GK,3,224,0,0,0,0,0,5,2,7.20,Hearts Of Oak SC,0.100000,14,Ghana
Luis Ricardo Mejia,1223,GK,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,0,0,5.93,Club Nacional,11.521400,56,Panama


In [97]:
for seleccion in jugadores["seleccion_nacional"].unique():
    num_porteros = len(jugadores.loc[((jugadores["posicion"]=="GK")&(jugadores["seleccion_nacional"]==seleccion))])
    print(seleccion, num_porteros)

Mexico 3
South Africa 3
South Korea 3
Czechia 3
Canada 3
Bosnia and Herzegovina 3
Qatar 3
Switzerland 3
Brazil 3
Morocco 3
Haiti 3
Scotland 3
USA 3
Paraguay 3
Australia 3
Türkiye 3
Germany 3
Curaçao 3
Côte d'Ivoire 3
Ecuador 3
Netherlands 3
Japan 3
Sweden 3
Tunisia 3
Belgium 3
Egypt 4
IR Iran 3
New Zealand 3
Spain 3
Cabo Verde 3
Saudi Arabia 3
Uruguay 3
France 3
Senegal 3
Iraq 3
Norway 3
Argentina 3
Algeria 3
Austria 3
Jordan 3
Portugal 3
Congo DR 3
Uzbekistan 3
Colombia 3
England 3
Croatia 3
Ghana 3
Panama 3


In [98]:
jugadores.loc[((jugadores["posicion"]=="GK")&(jugadores["seleccion_nacional"]=="Egypt"))]

,player_id,posicion,partidos_jugados,minutos_jugados,goles,asistencias,tiros,tarjetas_amarillas,tarjetas_rojas,paradas,goles_concedidos,calificacion_media,club_jugador,valor_mercado,partidos_seleccion,seleccion_nacional
nombre_jugador,,,,,,,,,,,,,,,,
Elsayed Mohamed Mohamed Elshenawy,651,GK,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,0,0,5.93,Al Ahly FC,3.2268,77,Egypt
Mohamed Soliman Mahdy Soliman,666,GK,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,0,0,5.93,Zamalek SC,6.7568,0,Egypt
Abdelaziz Mohame Mostafa,673,GK,5,480,0,0,9,1,0,11,7,7.18,Al Ahly FC,7.3868,11,Egypt
Alaaeldin Soliman Mohamed Alaa,676,GK,0,0,<NA>,<NA>,<NA>,<NA>,<NA>,0,0,5.93,El Gouna FC,3.1100,0,Egypt


como curiosidad, hemos observado que hay 48 selecciones y en teoria cada una debería llevar 3 porteros lo cual da un total de 144 porteros. Pero como observabamos que salían seleccionados 145 porteros, uno más de lo que debería, hemos buscado y es Egipto la unica seleccion que llevó 4 porteros en lugar de 3. Tras verificarlo con las fuentes oficiales del mundial, este dato es correcto.

In [99]:
jugadores

,player_id,posicion,partidos_jugados,minutos_jugados,goles,asistencias,tiros,tarjetas_amarillas,tarjetas_rojas,paradas,goles_concedidos,calificacion_media,club_jugador,valor_mercado,partidos_seleccion,seleccion_nacional
nombre_jugador,,,,,,,,,,,,,,,,
José Raúl Rangel,1,GK,5,438,0,0,0,0,0,7,3,7.200000,CD Guadalajara,8.933600,15,Mexico
Jorge Eduardo Sanchez,2,DEF,4,349,0,1,1,1,0,<NA>,<NA>,6.680000,PAOK Saloniki,24.628600,59,Mexico
César Jasib Montes,3,DEF,4,315,0,0,1,0,1,<NA>,<NA>,7.050000,FC Lokomotiv Moscow,10.912600,69,Mexico
Edson Omar Alvarez,4,DEF,4,239,0,0,2,1,0,<NA>,<NA>,7.080000,Fenerbahçe SK,10.875900,100,Mexico
Johan Felipe Vasquez,5,DEF,4,360,0,0,1,1,0,<NA>,<NA>,7.080000,Genoa CFC,18.265400,47,Mexico
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Orlando Mosquera,1244,GK,3,270,0,0,0,0,0,8,4,6.800000,Al Fayha FC,0.700000,50,Panama
Michael Amir Murillo,1245,DEF,3,270,0,0,5,0,0,<NA>,<NA>,6.930000,Be ş ikta ş JK,7.830200,94,Panama
Azarías Emmanuel Londono,1246,FWD,3,53,0,0,1,0,0,<NA>,<NA>,6.570000,CD Universidad Católica,4.490400,13,Panama


observamos que ya no queda ingun valor de calificacion_media nulo

In [100]:
print(jugadores["calificacion_media"].isna().sum())

0


### Imputacion por interpolacion de los valores nulos de minuto_evento

In [101]:
eventos["minuto_evento"].isna().sum()

np.int64(38)

vamos a interpolar los ausentes, pero agrupando segun el partido (que no se interpole un valor con los minutos de partido de dos partidos diferentes...)
Para ello necesitamos que estén ordenados de manera creciente por index_partido y minuto_partido, que ya lo está

In [102]:
eventos

,index_evento,index_partido,minuto_evento,tipo_evento,nombre_jugador,seleccion_evento
0,1,1,9,Goal,Julián Andrés Quinones,Mexico
1,2,1,9,Assist,Erik Antonio Lira,Mexico
2,3,1,17,Yellow Card,Teboho Mokoena,South Africa
3,4,1,23,Yellow Card,Brian Gutierrez,Mexico
4,5,1,55,Red Card,Sphephelo S'miso Sithole,South Africa
...,...,...,...,...,...,...
819,830,104,93,Red Card,Enzo Jeremías Fernandez,Argentina
820,831,104,106,Goal,Ferran Torres,Spain
821,832,104,106,Assist,Nicholas Williams,Spain
822,833,104,111,Yellow Card,Alexis Mac Allister,Argentina


In [103]:
eventos.head(20)

,index_evento,index_partido,minuto_evento,tipo_evento,nombre_jugador,seleccion_evento
0,1,1,9,Goal,Julián Andrés Quinones,Mexico
1,2,1,9,Assist,Erik Antonio Lira,Mexico
2,3,1,17,Yellow Card,Teboho Mokoena,South Africa
3,4,1,23,Yellow Card,Brian Gutierrez,Mexico
4,5,1,55,Red Card,Sphephelo S'miso Sithole,South Africa
5,6,1,55,VAR Review,Sphephelo S'miso Sithole,South Africa
6,7,1,67,Goal,Raúl Alonso Jimenez,Mexico
7,8,1,67,Assist,Roberto Carlos Alvarado,Mexico
8,9,1,74,Yellow Card,Nkosinathi Emmanuel Sibisi,South Africa
9,10,1,84,Red Card,Themba Zwane,South Africa


In [104]:
eventos["minuto_evento"] = eventos.groupby("index_partido")["minuto_evento"].transform(lambda x: x.interpolate())

In [105]:
eventos.head(20)

,index_evento,index_partido,minuto_evento,tipo_evento,nombre_jugador,seleccion_evento
0,1,1,9.0,Goal,Julián Andrés Quinones,Mexico
1,2,1,9.0,Assist,Erik Antonio Lira,Mexico
2,3,1,17.0,Yellow Card,Teboho Mokoena,South Africa
3,4,1,23.0,Yellow Card,Brian Gutierrez,Mexico
4,5,1,55.0,Red Card,Sphephelo S'miso Sithole,South Africa
5,6,1,55.0,VAR Review,Sphephelo S'miso Sithole,South Africa
6,7,1,67.0,Goal,Raúl Alonso Jimenez,Mexico
7,8,1,67.0,Assist,Roberto Carlos Alvarado,Mexico
8,9,1,74.0,Yellow Card,Nkosinathi Emmanuel Sibisi,South Africa
9,10,1,84.0,Red Card,Themba Zwane,South Africa


In [106]:
eventos["minuto_evento"].isna().sum()

np.int64(1)

ATENCIÓN observamos que tras el interpolate solo queda 1 valor de minuto como nan, esto se debe a que es el primer evento del partido con indice 3, por lo que no tiene un valor previo en el mismo partido e interpolate no puede inventarse un valor de minuto previo.
Aquí podemos decidir poner un bfill además del interpolate para rellenar ese último dato. Lo hacemos para que no quede ningún nulo, como pide el enunciado, PERO sabiendo que es una imputación muy poco precisa!!

In [107]:
eventos["minuto_evento"] = eventos.groupby("index_partido")["minuto_evento"].transform(lambda x: x.bfill())
eventos["minuto_evento"] = eventos["minuto_evento"].round().astype("Int64")   #el minuto es un entero, la interpolacion no puede dejar minutos decimales...
eventos["minuto_evento"].isna().sum()

np.int64(0)

In [108]:
partidos

,marcador_local,marcador_visitante,penalty_local,penalty_visitante,fecha_partido,arbitro_partido,ronda,estadio,ciudad_partido,pais_partido,prorroga,seleccion_local,seleccion_visitante
index_partido,,,,,,,,,,,,,
1,2,0,0,0,2026-06-11,Wilton Sampaio,Group Stage,Mexico City Stadium (Estadio Azteca),Mexico City,MEX,NO,Mexico,South Africa
2,2,1,0,0,2026-06-11,Amin Omar,Group Stage,Guadalajara Stadium (Estadio Akron),Zapopan,MEX,NO,South Korea,Czechia
3,1,1,0,0,2026-06-12,Facundo Tello,Group Stage,Toronto Stadium (BMO Field),Toronto,CAN,NO,Canada,Bosnia and Herzegovina
4,4,1,0,0,2026-06-12,Danny Makkelie,Group Stage,Los Angeles Stadium (SoFi Stadium),Inglewood,USA,NO,USA,Paraguay
5,1,1,0,0,2026-06-13,Saíd Martínez,Group Stage,Boston Stadium (Gillette Stadium),Foxborough,USA,NO,Qatar,Switzerland
...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,3,1,0,0,2026-07-12,João Pinheiro,Quarter-finals,Kansas City Stadium (Arrowhead Stadium),Kansas City,USA,SI,Argentina,Switzerland
101,0,2,0,0,2026-07-14,Iván Barton,Semi-finals,Dallas Stadium (AT&T Stadium),Arlington,USA,NO,France,Spain
102,1,2,0,0,2026-07-15,Ismail Elfath,Semi-finals,Atlanta Stadium (Mercedes-Benz Stadium),Atlanta,USA,NO,England,Argentina


#### ya tenemos los 3 df pedidos: partidos, jugadores y eventos, listos para comenzar el proyecto!

In [109]:
jugadores.drop(columns=["player_id"], inplace=True)
#reordeno las columnas para que salgan igual que en el enunciado
jugadores = jugadores[["seleccion_nacional", "club_jugador", "partidos_seleccion", "posicion", "valor_mercado",
                       "partidos_jugados", "minutos_jugados", "goles", "asistencias", "tiros",
                       "tarjetas_amarillas", "tarjetas_rojas", "paradas", "goles_concedidos", "calificacion_media"]]
jugadores

,seleccion_nacional,club_jugador,partidos_seleccion,posicion,valor_mercado,partidos_jugados,minutos_jugados,goles,asistencias,tiros,tarjetas_amarillas,tarjetas_rojas,paradas,goles_concedidos,calificacion_media
nombre_jugador,,,,,,,,,,,,,,,
José Raúl Rangel,Mexico,CD Guadalajara,15,GK,8.933600,5,438,0,0,0,0,0,7,3,7.200000
Jorge Eduardo Sanchez,Mexico,PAOK Saloniki,59,DEF,24.628600,4,349,0,1,1,1,0,<NA>,<NA>,6.680000
César Jasib Montes,Mexico,FC Lokomotiv Moscow,69,DEF,10.912600,4,315,0,0,1,0,1,<NA>,<NA>,7.050000
Edson Omar Alvarez,Mexico,Fenerbahçe SK,100,DEF,10.875900,4,239,0,0,2,1,0,<NA>,<NA>,7.080000
Johan Felipe Vasquez,Mexico,Genoa CFC,47,DEF,18.265400,4,360,0,0,1,1,0,<NA>,<NA>,7.080000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Orlando Mosquera,Panama,Al Fayha FC,50,GK,0.700000,3,270,0,0,0,0,0,8,4,6.800000
Michael Amir Murillo,Panama,Be ş ikta ş JK,94,DEF,7.830200,3,270,0,0,5,0,0,<NA>,<NA>,6.930000
Azarías Emmanuel Londono,Panama,CD Universidad Católica,13,FWD,4.490400,3,53,0,0,1,0,0,<NA>,<NA>,6.570000


In [110]:
#reordeno las columnas para que salgan igual que en el enunciado
eventos = eventos.set_index("index_evento")[["minuto_evento", "index_partido", "nombre_jugador", "tipo_evento", "seleccion_evento"]]
eventos

,minuto_evento,index_partido,nombre_jugador,tipo_evento,seleccion_evento
index_evento,,,,,
1,9,1,Julián Andrés Quinones,Goal,Mexico
2,9,1,Erik Antonio Lira,Assist,Mexico
3,17,1,Teboho Mokoena,Yellow Card,South Africa
4,23,1,Brian Gutierrez,Yellow Card,Mexico
5,55,1,Sphephelo S'miso Sithole,Red Card,South Africa
...,...,...,...,...,...
830,93,104,Enzo Jeremías Fernandez,Red Card,Argentina
831,106,104,Ferran Torres,Goal,Spain
832,106,104,Nicholas Williams,Assist,Spain


### Guardo los dfs ya limpios en excel (.xlsx) para trabajar con ellos proximamente

In [111]:
#permite crear una carpeta donde guardar los df limpios
import os
os.makedirs(RUTA_SALIDA, exist_ok=True)

partidos.to_excel(fr"{RUTA_SALIDA}partidosWC26.xlsx", index=True)
jugadores.to_excel(fr"{RUTA_SALIDA}jugadoresWC2026.xlsx", index=True)
eventos.to_excel(fr"{RUTA_SALIDA}eventosWC26.xlsx", index=True)